In [27]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel ,Field
from langchain.schema.runnable import RunnableParallel,RunnableBranch,RunnableLambda
from typing import Literal
load_dotenv()

True

In [ ]:
# Model
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

In [45]:
# Parsers
parser = StrOutputParser()


# Pydantic model with both fields
class Feedback(BaseModel):
    feedback: str = Field(description="The original feedback text")
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the feedback")

parser2 = PydanticOutputParser(pydantic_object=Feedback)

In [46]:
# Sentiment classification prompt
prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative:\n{feedback}\n{format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction': parser2.get_format_instructions()}
)

In [47]:
# Classifier chain (returns Feedback object)
classifier_chain = prompt1 | model | parser2

In [48]:
# Response prompts
prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback:\n{feedback}',
    input_variables=["feedback"]
)
prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback:\n{feedback}',
    input_variables=["feedback"]
)

In [49]:
# RunnableBranch - needs both sentiment & feedback
branch_chain = RunnableBranch(
    (lambda x: x["sentiment"] == "positive", prompt2 | model | parser),
    (lambda x: x["sentiment"] == "negative", prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)


In [50]:

# Combine: classify + forward feedback + sentiment
chain = classifier_chain | (lambda x: {"sentiment": x.sentiment, "feedback": x.feedback}) | branch_chain


In [51]:
# Test
print(chain.invoke({"feedback": "this is a terrible smartphone"}))

Okay, here are a few response options, ranging from basic to more proactive, along with explanations of why they're effective:

**Option 1: Basic Acknowledgment (Good for quick response, but less informative)**

> "We're sorry to hear you're having a negative experience with the smartphone. Could you please tell us more about what you dislike so we can understand the issue?"

**Why it works:**

*   **Acknowledges the feedback:** Shows you're not ignoring the comment.
*   **Asks for specifics:**  Opens the door for a more detailed explanation.
*   **Professional Tone:** Maintains a neutral and professional tone, even with negative feedback.

**Option 2: Slightly More Detailed (Good for showing you care and want to improve)**

> "We're very sorry to hear that you feel that way about the smartphone. We strive to create high-quality products, and we're disappointed that it didn't meet your expectations.  Could you please elaborate on what you found terrible about it? Knowing the specific i

In [52]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Lambda |         
        +--------+         
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *      